# Unit 02 · Building the processed dataset v1 (Claude's notebook)

This notebook is plumbing (`xray3d/prep.py`), but every idea in it is **your** code: `touches_scan_edge`
(screening), `bbox_center_mm` / `cube_grid` / `resample` / `occupancy` (the cube and the CT box), `hu_to_mu`
(attenuation) and `project_pinhole` (the geometry check). Run it after unit 02 part A is in `src/xray3d`.

* **Locally:** builds the 27 usable small-subset pelvises into `<data>/processed/v1` (~1 min each on a CPU).
* **On Kaggle** (GPU on, Internet on): streams the **full** TotalSegmentator (1,228 CTs, 23.6 GB) subject by
  subject out of the zip on Zenodo, screens them, builds the usable ones into `/kaggle/working/processed/v1`.
  Resumable: if the session ends, rerun the notebook and it continues.

Format: see `src/xray3d/dataset.py` (and the unit 02 contract in `docs/superpowers/specs`).

In [ ]:
# --- setup (written by Claude, no need to change it): the same cell works locally, on Colab and on Kaggle ---
import os, sys, subprocess
from pathlib import Path

UNIT = "02-shape-and-metrics"
REPO_URL = "https://github.com/AdirBuskila/pelvis-xray-3d"
IN_CLOUD = "google.colab" in sys.modules or bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))
if IN_CLOUD:
    root = Path("/content" if "google.colab" in sys.modules else "/kaggle/working") / "pelvis-xray-3d"
    if not root.exists():  # Kaggle: turn Internet on in the notebook settings first
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(root)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(root)], check=True)
else:  # local: the repo is the first folder above this notebook that has a pyproject.toml
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(root / "units" / UNIT)
for p in (root / "src", Path.cwd()):  # import xray3d (even before `pip install -e .`) and this unit's checks
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
import xray3d
from xray3d import data, env, plotting
import test_unit as T

env.summary()

In [ ]:
# DiffDRR renders the X-rays. Local: python -m pip install -e ".[drr]". Colab/Kaggle: installed here.
try:
    import diffdrr  # noqa: F401
except ImportError:
    if IN_CLOUD:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "diffdrr"], check=True)
        import diffdrr  # noqa: F401
    else:
        raise
from xray3d import dataset, prep, shape, volume, physics, geometry
need = [(shape, "bbox_center_mm"), (shape, "cube_grid"), (shape, "resample"), (shape, "occupancy"),
        (volume, "touches_scan_edge"), (physics, "hu_to_mu"), (geometry, "project_pinhole")]
missing = [f"{m.__name__}.{f}" for m, f in need if not hasattr(m, f)]
assert not missing, f"move these into src/xray3d first: {missing}"
OUT = Path("/kaggle/working/processed/v1") if env.where() == "kaggle" else env.processed_root() / "v1"
print("writing to", OUT, "| GPU:", env.has_gpu())

## 1 · Check the geometry first

Single bright voxels at known positions are rendered with DiffDRR and projected with `dataset.camera` + your
`project_pinhole`. They must agree to well under a pixel, or every image in the dataset would be misplaced.

In [ ]:
err = prep.check_geometry()
print(f"largest error {err:.2f} px")
assert err < 0.5, "DiffDRR and dataset.camera disagree: don't build until this is fixed"

## 2 · Build

`FULL = False` builds from the small subset on disk (the default locally). On Kaggle set `FULL = True`.
`LIMIT` builds only the first few subjects (try 5 first, look at them, then set it to `None`).

In [ ]:
FULL = env.where() == "kaggle"
LIMIT = None
if FULL:
    rec = prep.build_remote(OUT, limit=LIMIT)
else:
    ids = data.GOOD_SUBJECTS if LIMIT is None else data.GOOD_SUBJECTS[:LIMIT]
    rec = prep.build_local(OUT, ids)
print(prep.count_reasons(rec))

## 3 · Look at it

Every subject's occupancy should hold a few thousand voxels (an adult pelvis is ~700–1,400 cm³, i.e. 5,000–
11,000 voxels of 0.125 cm³). The silhouette IoU (projected occupancy vs. the pelvis-only DRR) must be ≥ 0.9.

In [ ]:
import os
os.environ["XRAY3D_PROCESSED"] = str(OUT.parent)     # read back what was just written
ids = dataset.subjects()
occ_sum, ious = [], []
for sid in ids:
    d = dataset.load_subject(sid)
    occ_sum.append(int(d["occ"].sum()))
    ious.append([prep.silhouette_iou(d, v) for v in dataset.VIEWS])
ious = np.array(ious)
print(f"{len(ids)} subjects; occupied voxels min {min(occ_sum)}, median {int(np.median(occ_sum))}, max {max(occ_sum)}")
print(f"silhouette IoU  ap: min {ious[:, 0].min():.3f} mean {ious[:, 0].mean():.3f}   lat: min {ious[:, 1].min():.3f} mean {ious[:, 1].mean():.3f}")
bad = [s for s, n, i in zip(ids, occ_sum, ious) if not (4000 < n < 14000) or i.min() < 0.9]
print("subjects to look at:", bad or "none")

In [ ]:
show = ids[:4]
fig, ax = plt.subplots(len(show), 5, figsize=(17, 3.5 * len(show)))
for r, sid in enumerate(show):
    d = dataset.load_subject(sid)
    for c, (img, name) in enumerate([(d["ap"], "ap"), (d["lat"], "lat"), (d["ap_bone"], "ap_bone"), (d["lat_bone"], "lat_bone"),
                                     (prep.coronal(d["occ"]), "occ, coronal layer")]):
        ax[r, c].imshow(img, cmap="gray", interpolation="nearest"); ax[r, c].set_title(f"{sid} {name}"); ax[r, c].axis("off")
plt.tight_layout(); plt.show()

## 4 · Keep it

The output is a folder of `.npz` files plus `index.json` and `screening.json` (a few hundred kB per subject).

**Kaggle → a Kaggle dataset** (what units 03–08 attach): when the run is done, *Save Version* (Save & Run All
is not needed if it already ran; "Quick save" keeps the output). Then open the notebook's **Output** tab →
*New Dataset*, name it **`pelvis-xray3d-v1`**, keep it **private** (TotalSegmentator is CC BY 4.0, but there's
no reason to publish it). In later notebooks: *Add input → Your datasets → pelvis-xray3d-v1*;
`dataset.root()` finds it under `/kaggle/input/`.

**Kaggle/local → Google Drive** (for Colab): the next cell writes `pelvis-xray3d-v1.zip` (holding `v1/...`).
Download it from the Output tab (Kaggle) and upload it to `MyDrive/pelvis-xray3d/pelvis-xray3d-v1.zip`;
`dataset.ensure_local()` on Colab mounts Drive and unzips it.

In [ ]:
import shutil
zip_path = shutil.make_archive(str(OUT.parent / "pelvis-xray3d-v1"), "zip", root_dir=OUT.parent, base_dir=OUT.name)
print("wrote", zip_path, f"({Path(zip_path).stat().st_size / 2**20:.0f} MB)")